<a href="https://colab.research.google.com/github/honordold/ds2002-fa26/blob/main/2026_09_30_%E2%80%94_JSON_to_Tidy_Tables_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

holdings = data['portfolio']['holdings']
print('portfolio name:', data['portfolio']['name'])
print('holdings:', len(holdings))
print('price records:', sum(len(h['prices']) for h in holdings))

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio name: Cavalier Growth
holdings: 3
price records: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: _3_
- price rows: _9_
- one row of the table you are about to build is one _ticker on one day_

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
print(data['portfolio']['holdings'][0]['company']['sector'])   # Technology

try:
    print(data['portfolio']['holdings'][2]['company']['sector'])
except KeyError as e:
    print('KeyError:', e)   # KeyError: 'sector' -- KO's company object has no sector key

for h in holdings:
    company = h['company']
    sector = company['sector'] if 'sector' in company else 'unknown'
    print(h['ticker'], sector)

Technology
KeyError: 'sector'
AAPL Technology
MSFT Technology
KO unknown


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.


In [5]:
for h in holdings:
    if 'sector' not in h['company']:
        print('missing field: sector is missing for', h['ticker'])
    for p in h['prices']:
        if p['close'] is None:
            print('null close:', h['ticker'], p['date'])
        if isinstance(p['close'], str):
            print('string close:', h['ticker'], p['date'], repr(p['close']))
        if p['volume'] == 0:
            print('zero volume:', h['ticker'], p['date'])

string close: MSFT 2026-09-29 '498.75'
zero volume: MSFT 2026-09-29
missing field: sector is missing for KO
null close: KO 2026-09-29


**TODO:** your answer to the last question: _KO's null close and KO's missing sector are genuinely missing data. MSFT's "498.75" is a real measurement stored as the wrong type, so you convert it rather than throw it out. MSFT's zero volume is a recorded value, not an absence, so it is treated as a measurement._

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
px = pd.json_normalize(holdings, record_path='prices',
                       meta=['ticker', 'shares', ['company', 'name'], ['company', 'sector']],
                       sep='.', errors='ignore')
assert len(px) == 9
print(px.dtypes)

date              object
close             object
volume             int64
ticker            object
shares            object
company.name      object
company.sector    object
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
px['close'] = pd.to_numeric(px['close'], errors='coerce')
print(px['close'].dtype)            # float64
print(px['close'].isna().sum())     # 1

float64
1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
# 1
print(px.groupby('ticker')['close'].agg(['mean', 'count']).round(2))

# 2
last = px[px['date'] == '2026-09-30'].copy()
last['value'] = last['shares'] * last['close']
print(last[['ticker', 'shares', 'close', 'value']])
print('total:', last['value'].sum())
print('largest:', last.loc[last['value'].idxmax(), 'ticker'])

# 3
with_zero = px['volume'].mean()
without_zero = px.loc[px['volume'] != 0, 'volume'].mean()
print('all 9 rows:', with_zero)
print('8 rows, zero excluded:', without_zero)
print('difference:', without_zero - with_zero)

          mean  count
ticker               
AAPL    231.45      3
KO       71.08      2
MSFT    502.05      3
  ticker shares  close    value
2   AAPL    120  233.1  27972.0
5   MSFT     45  505.3  22738.5
8     KO    300   70.9  21270.0
total: 71980.5
largest: AAPL
all 9 rows: 26677777.777777776
8 rows, zero excluded: 30012500.0
difference: 3334722.222222224


**TODO:** the zero-volume question: _Dropping the zero was not the right call. Zero is a recorded measurement, not a missing value, and silently removing it inflates the mean by about 3.3 million shares. If you think it's bad data, flag it and report both numbers instead of quietly dropping it._

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
summary = px.groupby('ticker')['close'].agg(['mean', 'count']).round(2)
result = {
    'as_of': data['as_of'],
    'currency': data['currency'],
    'portfolio': data['portfolio']['name'],
    'tickers': {t: {'mean_close': float(r['mean']), 'days': int(r['count'])}
                for t, r in summary.iterrows()}
}
with open('portfolio_summary.json', 'w') as f:
    json.dump(result, f, indent=2, allow_nan=False)
print(open('portfolio_summary.json').read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "portfolio": "Cavalier Growth",
  "tickers": {
    "AAPL": {
      "mean_close": 231.45,
      "days": 3
    },
    "KO": {
      "mean_close": 71.08,
      "days": 2
    },
    "MSFT": {
      "mean_close": 502.05,
      "days": 3
    }
  }
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [12]:
# A
vendors = pd.json_normalize(response, record_path=['zones', 'vendors'],
                            meta=['game', 'date', ['zones', 'zone'], ['zones', 'capacity']],
                            sep='_')
print(len(vendors))   # 4

# B -- Stadium Dogs (V-07) has no revenue
# Don't fill with 0: they took 210 orders, so revenue wasn't zero, it just wasn't reported.
# A 0 would make it look like they made no money and would drag down any revenue total or average.
vendors['revenue_reported'] = vendors['sales_revenue'].notna()

# C
lines = pd.json_normalize(response, record_path=['zones', 'vendors', 'lines'],
                          meta=[['zones', 'vendors', 'vendor_id'], ['zones', 'zone']],
                          sep='_')
print(len(lines), lines['qty'].sum())   # 6 755

# D
joined = lines.merge(vendors[['vendor_id', 'name', 'zones_capacity']],
                     left_on='zones_vendors_vendor_id', right_on='vendor_id',
                     how='left', indicator=True)
print(len(joined))                       # 6
print(joined['_merge'].value_counts())   # both 6, left_only 0, right_only 0

4
6 755
6
_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [ ]:
# TODO A, B, C, D

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [13]:
holdings_count = 3
price_rows = 9
largest_position = 'AAPL'
null_vs_zero = "KO's close on 9/29 was missing (null); MSFT's volume of 0 on 9/29 was a real recorded measurement"
zero_volume_call = 'Kept it: a zero is a measurement, not missing data, and dropping it inflates the mean by ~3.3M shares'

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: KO's close on 9/29 was missing (null); MSFT's volume of 0 on 9/29 was a real recorded measurement
zero volume: Kept it: a zero is a measurement, not missing data, and dropping it inflates the mean by ~3.3M shares
